In [4]:
import refinitiv.data as rd
import pandas as pd
import numpy as np
import os

# Default settings markets

In [5]:
%load_ext autoreload
%autoreload 2

In [6]:
markets = ["XPAR", "XAMS", "XBRU", "XMAD", "XNYS", "XNAS"]
ESG_SCORE_THRESH_HOLD = 50
MARKET_VALUE_THRESH_HOLD = 1e9

# STOCK SCREENER 

### ESG Score Screening and Market Value Screening

In [7]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x12b231790 {name='workspace'}>

In [8]:
rd.close_session()

In [9]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x12b231790 {name='workspace'}>

In [10]:
mic_codes = ", ".join(markets)
screener_query = (
    "SCREEN("
    "U(IN(Equity(active,public,primary))), "
    f"IN(TR.ExchangeMarketIdCode, {mic_codes}), "
    "TR.TRESGScore > 50"
    ")"
)

In [11]:
df_screener = rd.get_data(screener_query,
                          fields =[
                              "TR.CommonName",
                              "TR.TRESGScore",
                              "TR.CompanyMarketCap(CURN=USD)"
                          ]
)

In [12]:
df_screener = df_screener[df_screener["Company Market Cap"] >= MARKET_VALUE_THRESH_HOLD]
df_screener["Company Market Cap"].min()
df_screener.head()


,Instrument,Company Common Name,ESG Score,Company Market Cap
0,FN.N,Fabrinet,58.649173,19547804218.709999
1,TGLS.N,Tecnoglass Inc,70.468723,2038250796.56
3,AMA.MC,Amadeus IT Group SA,89.667454,28028983061.450802
4,ZWS.N,Zurn Elkay Water Solutions Corp,69.603064,8503360714.52
5,EDEN.PA,Edenred SE,79.837036,5479375024.24292


In [13]:
df_screener.head()

,Instrument,Company Common Name,ESG Score,Company Market Cap
0,FN.N,Fabrinet,58.649173,19547804218.709999
1,TGLS.N,Tecnoglass Inc,70.468723,2038250796.56
3,AMA.MC,Amadeus IT Group SA,89.667454,28028983061.450802
4,ZWS.N,Zurn Elkay Water Solutions Corp,69.603064,8503360714.52
5,EDEN.PA,Edenred SE,79.837036,5479375024.24292


# Fetch Data for the selected stocks

### Settings for historical data retrieval

In [14]:
selected_stocks = df_screener.sort_values(by="Company Market Cap", ascending=False)["Instrument"].tolist()
selected_stocks_by_name = df_screener[["Company Common Name", "Instrument"]]
pd.DataFrame(selected_stocks).to_csv("selected_stocks.csv", index=False)
selected_stocks[:5]


['LLY.N', 'JPM.N', 'XOM.N', 'V.N', 'JNJ.N']

In [15]:
# Rolling window of 7 months
rolling_window = 12 * 21  # Assuming 21 trading days in a month

In [16]:
END_DATE = pd.Timestamp.today()
START_DATE = END_DATE - pd.Timedelta(days=rolling_window)


### Fetch close price history for the selected stocks

In [17]:
# Import necesaary library
from datetime import datetime, timedelta
import pandas as pd

In [18]:

df_history = rd.get_history(
    universe=selected_stocks[:300],
    interval='1d',
    fields=['TRDPRC_1'],
    start=START_DATE,
    end=END_DATE
)
df_history.to_pickle("stock_history.pkl")

/opt/anaconda3/envs/final_project/lib/python3.12/site-packages/refinitiv/data/_tools/_dataframe.py:177:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [19]:
import os
from datetime import timedelta

file_path = "stock_history.pkl"

if os.path.exists(file_path):
    old_df = pd.read_pickle(file_path)
    
    last_date = old_df.index.get_level_values(0).max()
    new_start = last_date + timedelta(days=1)

    df_new = rd.get_history(
        universe=selected_stocks[:300],
        interval='1d',
        fields=['TRDPRC_1'],
        start=new_start,
        end=END_DATE
    )

    df_history = pd.concat([old_df, df_new])
    df_history = df_history[~df_history.index.duplicated()]
    
else:
    df_history = rd.get_history(
        universe=selected_stocks[:300],
        interval='1d',
        fields=['TRDPRC_1'],
        start=START_DATE,
        end=END_DATE
    )

df_history.to_pickle(file_path)

### Upload the Start and End date to the database

In [20]:
df_history.shape
df_history.head()

TRDPRC_1,LLY.N,JPM.N,XOM.N,V.N,JNJ.N,ASML.AS,MA.N,ABBV.N,PG.N,HD.N,...,EQR.N,LH.N,BG.N,PERP.PA,DGX.N,IP.N,HUM.N,TSN.N,NI.N,PUBP.PA
Date,,,,,,,,,,,,,,,,,,,,,
2025-06-23,770.64,278.27,111.74,343.75,151.32,668.7,542.35,183.76,161.03,356.96,...,69.06,261.04,84.15,86.84,179.72,46.09,234.66,55.11,40.51,93.36
2025-06-24,778.08,281.26,108.34,351.63,152.19,691.4,557.53,185.55,160.36,360.42,...,67.54,262.53,83.59,87.02,179.9,46.59,238.79,55.51,40.52,95.0
2025-06-25,792.3,284.06,108.37,345.26,152.28,694.5,549.7,185.39,158.97,361.86,...,65.64,258.7,82.22,85.32,178.65,46.4,238.51,54.64,39.73,93.64
2025-06-26,795.12,288.75,109.99,346.03,152.01,677.4,545.81,186.79,158.63,363.5,...,67.46,256.98,82.3,85.72,177.06,46.51,239.9,55.02,39.61,93.58
2025-06-27,775.45,287.11,109.38,348.61,152.41,682.5,550.32,182.31,159.86,368.74,...,67.35,260.59,80.44,86.02,177.83,47.37,241.88,55.24,39.97,95.78


In [21]:
df_history.isna().sum()

TRDPRC_1
LLY.N      5
JPM.N      5
XOM.N      5
V.N        5
JNJ.N      5
          ..
IP.N       5
HUM.N      5
TSN.N      5
NI.N       5
PUBP.PA    1
Length: 300, dtype: int64

### Cleaning data

In [22]:
# Fill missing values with the previous day's price
df_history = df_history.fillna(method='ffill')
print(df_history.isna().sum())
df_history.to_csv("stock_history.csv")

TRDPRC_1
LLY.N      0
JPM.N      0
XOM.N      0
V.N        0
JNJ.N      0
          ..
IP.N       0
HUM.N      0
TSN.N      0
NI.N       0
PUBP.PA    0
Length: 300, dtype: int64


/var/folders/d3/4j2xrjxn06v329lnxwrwmzs80000gn/T/ipykernel_43047/215517627.py:2:FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.


In [23]:
# if Date is a column -> set it as index; if not, just ensure index is datetime
df_history.index = pd.to_datetime(df_history.index, errors="coerce")
df_history = df_history[~df_history.index.isna()]
df_history.head()


TRDPRC_1,LLY.N,JPM.N,XOM.N,V.N,JNJ.N,ASML.AS,MA.N,ABBV.N,PG.N,HD.N,...,EQR.N,LH.N,BG.N,PERP.PA,DGX.N,IP.N,HUM.N,TSN.N,NI.N,PUBP.PA
Date,,,,,,,,,,,,,,,,,,,,,
2025-06-23,770.64,278.27,111.74,343.75,151.32,668.7,542.35,183.76,161.03,356.96,...,69.06,261.04,84.15,86.84,179.72,46.09,234.66,55.11,40.51,93.36
2025-06-24,778.08,281.26,108.34,351.63,152.19,691.4,557.53,185.55,160.36,360.42,...,67.54,262.53,83.59,87.02,179.9,46.59,238.79,55.51,40.52,95.0
2025-06-25,792.3,284.06,108.37,345.26,152.28,694.5,549.7,185.39,158.97,361.86,...,65.64,258.7,82.22,85.32,178.65,46.4,238.51,54.64,39.73,93.64
2025-06-26,795.12,288.75,109.99,346.03,152.01,677.4,545.81,186.79,158.63,363.5,...,67.46,256.98,82.3,85.72,177.06,46.51,239.9,55.02,39.61,93.58
2025-06-27,775.45,287.11,109.38,348.61,152.41,682.5,550.32,182.31,159.86,368.74,...,67.35,260.59,80.44,86.02,177.83,47.37,241.88,55.24,39.97,95.78


# Analyst the stock data

### Change in returns 

In [24]:
df_ret = np.log(df_history).diff().dropna()
df_ret.head()

TRDPRC_1,LLY.N,JPM.N,XOM.N,V.N,JNJ.N,ASML.AS,MA.N,ABBV.N,PG.N,HD.N,...,EQR.N,LH.N,BG.N,PERP.PA,DGX.N,IP.N,HUM.N,TSN.N,NI.N,PUBP.PA
Date,,,,,,,,,,,,,,,,,,,,,
2025-06-24,0.009608,0.010688,-0.0309,0.022665,0.005733,0.033383,0.027605,0.009694,-0.004169,0.009646,...,-0.022256,0.005692,-0.006677,0.002071,0.001001,0.01079,0.017447,0.007232,0.000247,0.017414
2025-06-25,0.018111,0.009906,0.000277,-0.018282,0.000591,0.004474,-0.014144,-0.000863,-0.008706,0.003987,...,-0.028535,-0.014696,-0.016525,-0.019729,-0.006973,-0.004086,-0.001173,-0.015797,-0.019689,-0.014419
2025-06-26,0.003553,0.016376,0.014838,0.002228,-0.001775,-0.02493,-0.007102,0.007523,-0.002141,0.004522,...,0.02735,-0.006671,0.000973,0.004677,-0.00894,0.002368,0.005811,0.006931,-0.003025,-0.000641
2025-06-27,-0.02505,-0.005696,-0.005561,0.007428,0.002628,0.007501,0.008229,-0.024276,0.007724,0.014312,...,-0.001632,0.01395,-0.02286,0.003494,0.004339,0.018322,0.00822,0.003991,0.009048,0.023237
2025-06-30,0.005248,0.009705,-0.01455,0.018305,0.002228,-0.007205,0.020895,0.017993,-0.003384,-0.005711,...,0.002077,0.007341,-0.001991,-0.016646,0.010071,-0.011465,0.010692,0.012592,0.009214,-0.001045


### Caculate the volatility of each stock

In [28]:
# Volatility calculator of stock
vol_20 = df_ret.rolling(window=20).std() * np.sqrt(252)
vol_60 = df_ret.rolling(window=60).std() * np.sqrt(252)

In [34]:
vol_20_mean = vol_20.mean(axis=1)
vol_60_mean = vol_60.mean(axis=1)

df_ret['Vol_Regime'] = (vol_20_mean > vol_60_mean).astype(int)

In [27]:
df_ret

TRDPRC_1,LLY.N,JPM.N,XOM.N,V.N,JNJ.N,ASML.AS,MA.N,ABBV.N,PG.N,HD.N,...,EQR.N,LH.N,BG.N,PERP.PA,DGX.N,IP.N,HUM.N,TSN.N,NI.N,PUBP.PA
Date,,,,,,,,,,,,,,,,,,,,,
2025-06-24,0.009608,0.010688,-0.0309,0.022665,0.005733,0.033383,0.027605,0.009694,-0.004169,0.009646,...,-0.022256,0.005692,-0.006677,0.002071,0.001001,0.01079,0.017447,0.007232,0.000247,0.017414
2025-06-25,0.018111,0.009906,0.000277,-0.018282,0.000591,0.004474,-0.014144,-0.000863,-0.008706,0.003987,...,-0.028535,-0.014696,-0.016525,-0.019729,-0.006973,-0.004086,-0.001173,-0.015797,-0.019689,-0.014419
2025-06-26,0.003553,0.016376,0.014838,0.002228,-0.001775,-0.02493,-0.007102,0.007523,-0.002141,0.004522,...,0.02735,-0.006671,0.000973,0.004677,-0.00894,0.002368,0.005811,0.006931,-0.003025,-0.000641
2025-06-27,-0.02505,-0.005696,-0.005561,0.007428,0.002628,0.007501,0.008229,-0.024276,0.007724,0.014312,...,-0.001632,0.01395,-0.02286,0.003494,0.004339,0.018322,0.00822,0.003991,0.009048,0.023237
2025-06-30,0.005248,0.009705,-0.01455,0.018305,0.002228,-0.007205,0.020895,0.017993,-0.003384,-0.005711,...,0.002077,0.007341,-0.001991,-0.016646,0.010071,-0.011465,0.010692,0.012592,0.009214,-0.001045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-23,0.047435,-0.043132,0.023354,-0.046002,0.01372,-0.00511,-0.059444,0.02056,0.026938,-0.013856,...,0.011436,0.009261,-0.005179,-0.035634,0.024499,-0.053913,-0.046748,-0.014194,-0.003673,-0.029861
2026-02-24,-0.015624,-0.001244,-0.009999,0.002281,0.001788,0.011303,0.003964,-0.004542,0.000666,0.019673,...,0.003783,0.006394,0.009761,0.004999,0.010121,-0.008821,-0.036654,0.00728,0.004751,0.011062
2026-02-25,-0.012864,0.019981,-0.001341,0.018607,-0.004517,0.019595,0.022614,-0.006676,-0.011501,-0.023447,...,-0.00252,-0.015266,-0.009267,-0.082638,-0.008339,-0.008212,0.008098,-0.012854,0.002367,-0.000825


# Strategy Define

The momentum will choose the top stock with the highest price change over the last month with the positive 60 day returns

- 30% of the funds will be fall within this portfolio
- 70% of the funds will be using adjusted-risk return strategy for making sure the stability of the portfolio

### Momentum Strategy

In [ ]:
# Copy the returns change DataFrame to calculate momentum
df_mom = df_ret.copy()

In [96]:
# Calculate the  momentum of 20 day and 60 day
mom20 = np.exp(df_mom.rolling(window=20).sum()) - 1
mom60 = np.exp(df_mom.rolling(window=60).sum()) - 1


In [97]:
# Clean all the N/A values
mom20 = mom20.dropna(how="all")
mom60 = mom60.dropna(how="all")

In [98]:
# Momentum calculator toda
mom20 = df_history / df_history.shift(20) - 1
mom60 = df_history / df_history.shift(60) - 1

In [99]:
# Today snapshot of momentum
mom20_t = mom20.iloc[-1]
mom60_t = mom60.iloc[-1]

In [100]:
# Calculate the z-score of momentum
z_cs_20 = (mom20_t - mom20_t.mean()) / mom20_t.std()
z_cs_60 = (mom60_t - mom60_t.mean()) / mom60_t.std()

In [101]:
stocks_data = {
    "Momentum 20": mom20_t,
    "Momentum 60": mom60_t,
    "Z-Score 20": z_cs_20,
    "Z-Score 60": z_cs_60
}
stocks_df = pd.DataFrame(stocks_data)
stocks_df.isna().sum()

Momentum 20    0
Momentum 60    0
Z-Score 20     0
Z-Score 60     0
dtype: int64

In [104]:
stocks_df["Score"] = 0.7*stocks_df["Z-Score 20"] + 0.3*stocks_df["Z-Score 60"]
top = stocks_df[(stocks_df["Momentum 60"]>0) & (stocks_df["Momentum 20"]>0)] \
        .sort_values("Score", ascending=False).head(20)

top

,Momentum 20,Momentum 60,Z-Score 20,Z-Score 60,Score
TRDPRC_1,,,,,
TPL.N,0.505024,0.719512,4.264858,3.575755,4.058127
GLW.N,0.456465,0.79988,3.808821,4.043755,3.879301
CIEN.N,0.384774,0.803465,3.135539,4.064631,3.414267
KEYS.N,0.420654,0.487489,3.4725,2.224648,3.098144
AU.N,0.375794,0.529264,3.051203,2.467913,2.876216
VRT.N,0.369051,0.424922,2.987879,1.860311,2.649609
FIX.N,0.251528,0.505709,1.884159,2.330751,2.018137
HWM.N,0.261678,0.351228,1.979486,1.431178,1.814993
MT.AS,0.206612,0.515847,1.462328,2.389784,1.740565


In [105]:
def mom_rule(df, top_k=6):
    x = df.copy()

    # 1) Filter trend
    x = x[x["Momentum 60"] > 0]

    # 2) Rank by Mom20 and take Top K
    picks = x.sort_values("Momentum 20", ascending=False).head(top_k)

    # 3) Size by z20 (not a buy filter)
    z = picks["Z-Score 20"]
    mult = np.where(z > 3.5, 0.3, np.where(z > 2.5, 0.6, 1.0))

    picks["size_mult"] = mult
    picks["weight"] = picks["size_mult"] / picks["size_mult"].sum()  # sum=1 within sleeve
    picks["investment"] = picks["weight"] * 2e7
    return picks[["Momentum 20","Momentum 60","Z-Score 20","weight","investment"]]

# usage:
picks = mom_rule(stocks_df, top_k=5)
picks

,Momentum 20,Momentum 60,Z-Score 20,weight,investment
TRDPRC_1,,,,,
TPL.N,0.505024,0.719512,4.264858,0.125,2500000.0
GLW.N,0.456465,0.79988,3.808821,0.125,2500000.0
KEYS.N,0.420654,0.487489,3.4725,0.250,5000000.0
CIEN.N,0.384774,0.803465,3.135539,0.250,5000000.0
AU.N,0.375794,0.529264,3.051203,0.250,5000000.0


### Min Variance 

In [106]:
ret = df_ret.copy()
ret = ret.replace([np.inf, -np.inf], np.nan).dropna(how="all")

In [53]:
ret

TRDPRC_1,LLY.N,JPM.N,XOM.N,V.N,JNJ.N,ASML.AS,MA.N,ABBV.N,PG.N,HD.N,...,LH.N,BG.N,PERP.PA,DGX.N,IP.N,HUM.N,TSN.N,NI.N,PUBP.PA,Vol_Regime
Date,,,,,,,,,,,,,,,,,,,,,
2025-06-24,0.009608,0.010688,-0.0309,0.022665,0.005733,0.033383,0.027605,0.009694,-0.004169,0.009646,...,0.005692,-0.006677,0.002071,0.001001,0.01079,0.017447,0.007232,0.000247,0.017414,0
2025-06-25,0.018111,0.009906,0.000277,-0.018282,0.000591,0.004474,-0.014144,-0.000863,-0.008706,0.003987,...,-0.014696,-0.016525,-0.019729,-0.006973,-0.004086,-0.001173,-0.015797,-0.019689,-0.014419,0
2025-06-26,0.003553,0.016376,0.014838,0.002228,-0.001775,-0.02493,-0.007102,0.007523,-0.002141,0.004522,...,-0.006671,0.000973,0.004677,-0.00894,0.002368,0.005811,0.006931,-0.003025,-0.000641,0
2025-06-27,-0.02505,-0.005696,-0.005561,0.007428,0.002628,0.007501,0.008229,-0.024276,0.007724,0.014312,...,0.01395,-0.02286,0.003494,0.004339,0.018322,0.00822,0.003991,0.009048,0.023237,0
2025-06-30,0.005248,0.009705,-0.01455,0.018305,0.002228,-0.007205,0.020895,0.017993,-0.003384,-0.005711,...,0.007341,-0.001991,-0.016646,0.010071,-0.011465,0.010692,0.012592,0.009214,-0.001045,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-23,0.047435,-0.043132,0.023354,-0.046002,0.01372,-0.00511,-0.059444,0.02056,0.026938,-0.013856,...,0.009261,-0.005179,-0.035634,0.024499,-0.053913,-0.046748,-0.014194,-0.003673,-0.029861,1
2026-02-24,-0.015624,-0.001244,-0.009999,0.002281,0.001788,0.011303,0.003964,-0.004542,0.000666,0.019673,...,0.006394,0.009761,0.004999,0.010121,-0.008821,-0.036654,0.00728,0.004751,0.011062,1
2026-02-25,-0.012864,0.019981,-0.001341,0.018607,-0.004517,0.019595,0.022614,-0.006676,-0.011501,-0.023447,...,-0.015266,-0.009267,-0.082638,-0.008339,-0.008212,0.008098,-0.012854,0.002367,-0.000825,1


In [ ]:
window = 60
ret_w = ret.iloc[-window:]